# Integrales-04 — Le rung elliptique du sunrise

Le carnet 02 intégrait le sunrise à masses égales en $d = 2$ par quatre voies
accordées — sans jamais nommer la structure cachée. Le carnet 03 faisait diverger
la bulle en $d = 4$. Celui-ci revient au sunrise et **exhibe sa courbe
elliptique** — non pas en la recopiant d'une référence, mais en la laissant
sortir du calcul : à un paramètre de Feynman fixé, l'intégrale interne se fait
en fonctions élémentaires, et ce qui reste vit sur la racine d'une **quartique**
$y^2 = b(\chi)\,c(\chi)$. Le classement BootLoops (polylogarithmes →
**elliptique** → K3 → Calabi-Yau) franchit ici son premier rang.

Le programme :

1. **rappel et ancre** — la forme paramétrique vérifiée du carnet 02,
   $I(s)=\iint 1/(U+s\,x_1x_2x_3)$, et l'ancre $I(0)=\frac{4}{\sqrt3}\mathrm{Cl}_2(\pi/3)$ ;
2. **la voie interne** — l'intégrale en $x$ faite à la main (un `artanh`),
   le reste donnant $I(s)$ comme intégrale **de $\log$ contre $d\chi/y$** :
   un dilogarithme elliptique en formation ;
3. **la courbe** — racines, dégénérescences ($s\to 0$ : une racine s'échappe
   à l'infini et la courbe devient le monde $\mathrm{Cl}_2$ du carnet 01 ;
   $s\to\infty$ : deux racines se collisionnent, pointe de la courbe) ;
4. **les périodes** — mesurées par quadrature, réduites à $K(k)$, module
   mesuré par birapport des racines ;
5. **$\tau(s)$ court** — la courbe dépend de l'échelle, contrairement au
   monde des constantes des carnets 01-02.

Comme toujours : outil réel (`mpmath`, 30 chiffres), chaque affirmation
quantitative vit dans une sortie commise, les refus sont documentés.

## 1. Rappel : la forme vérifiée du carnet 02

$$I(s) = \int_0^1\!\!\int_0^{1-x_1} \frac{dx_2\,dx_1}{U + s\,x_1x_2x_3},
\qquad U = x_1x_2+x_1x_3+x_2x_3, \quad x_3 = 1-x_1-x_2,$$

vérifiée au carnet 02 contre trois autres voies (rectangle $(u,v)$, Cheng–Wu,
hyperbolique) — avec l'ancre exacte $I(0) = \frac{4}{\sqrt3}\mathrm{Cl}_2(\pi/3)$,
le triangle du carnet 01 évalué à la cinématique $J(1,1,1)$.

In [1]:
from mpmath import mp, mpf, quad, sqrt, pi, log, atanh, ellipk, clsin
mp.dps = 30

def U(x1, x2, x3):
    return x1*x2 + x1*x3 + x2*x3

def I_A(s):
    # Sunrise d=2, voie A du carnet 02 (imbrication manuelle).
    def inner(x1):
        return quad(lambda x2: 1/(U(x1, x2, 1-x1-x2) + s*x1*x2*(1-x1-x2)), [0, 1-x1])
    return quad(inner, [0, 1])

anchor = (4/sqrt(3)) * clsin(2, pi/3)
val0 = I_A(0)
print("I(0)                        =", mp.nstr(val0, 22))
print("ancre (4/sqrt3) Cl2(pi/3)   =", mp.nstr(anchor, 22))
print("ecart relatif               =", mp.nstr(abs(val0 - anchor)/anchor, 3))

I(0)                        = 2.343907238689458890602
ancre (4/sqrt3) Cl2(pi/3)   = 2.343907238689458890602
ecart relatif               = 0.0


## 2. La voie interne : l'intégrale en $x$ se fait à la main

Fixons $x_3 = \chi$, $x_1 = x(1-\chi)$, $x_2 = (1-x)(1-\chi)$ — jacobien
$|dx_1\,dx_2| = (1-\chi)\,dx\,d\chi$. Le dénominateur devient

$$U + s\,x_1x_2x_3 = (1-\chi)\big[\chi + b(\chi)\,x(1-x)\big],
\qquad b(\chi) = (1-\chi)(1+s\chi),$$

et le facteur $(1-\chi)$ du jacobien **s'annule contre celui du dénominateur** :

$$I(s) = \int_0^1 d\chi \int_0^1 \frac{dx}{\chi + b(\chi)\,x(1-x)}.$$

L'intégrale interne est élémentaire — avec $c = b + 4\chi$ :

$$\int_0^1 \frac{dx}{a + b\,x(1-x)} = \frac{4}{\sqrt{b\,c}}\;\operatorname{artanh}\!\sqrt{\frac{b}{c}}
\qquad (a = \chi,\ 0 < b < c \text{ sur } \chi \in (0,1),\ s>0),$$

d'où

$$I(s) = \int_0^1 \frac{4\,\operatorname{artanh}\sqrt{b/c}}{\sqrt{b\,c}}\;d\chi
\qquad\text{avec}\qquad y^2 = b(\chi)\,c(\chi).$$

Deux erreurs de facteur ont été attrapées par le planted truth ci-dessous avant
d'être commises : le coefficient vaut $4$ (pas $2$), et le $(1-\chi)$ du jacobien
disparaît — il ne reste **pas** dans l'intégrande. La forme finale parle d'elle-même :
une intégrale **de $\log$ contre $d\chi/y$**, la différentielle de première espèce
de la courbe $y^2 = bc$ — exactement la manière dont un dilogarithme classique est
$\int \log(1-x)\,dx/x$. Le rung elliptique n'est pas plaqué : il sort du calcul.

In [2]:
def b_c(chi, s):
    b = (1-chi)*(1+s*chi)
    c = 1 + (3+s)*chi - s*chi*chi
    return b, c

def I_C(s):
    # Voie C : integrale interne analytique (artanh), reste en chi sur dchi/y.
    def f(chi):
        b, c = b_c(chi, s)
        return 4*atanh(sqrt(b/c))/sqrt(b*c)
    return quad(f, [0, mpf('0.5'), 1])

# planted truth de la formule interne : quad direct vs formule fermee, a=0.3, b=1.4
a0, b0 = mpf('0.3'), mpf('1.4')
c0 = b0 + 4*a0
lhs = quad(lambda x: 1/(a0 + b0*x*(1-x)), [0, 1])
rhs = 4*atanh(sqrt(b0/c0))/sqrt(b0*c0)
print("formule interne : ecart =", mp.nstr(abs(lhs-rhs)/rhs, 3))

print(f"{'s':>4} | {'voie A':>24} | {'voie C':>24} | |A-C|/A")
for s in [1, 9, 25]:
    a, c = I_A(s), I_C(s)
    print(f"{s:>4} | {mp.nstr(a, 22):>24} | {mp.nstr(c, 22):>24} | {mp.nstr(abs(a-c)/a, 3)}")

formule interne : ecart = 0.0
   s |                   voie A |                   voie C | |A-C|/A


   1 |  2.236792700212646510529 |  2.236792700212646510529 | 0.0


   9 |  1.697946461195426121942 |  1.697946461195426121942 | 0.0


  25 |  1.225709127276724404979 |  1.225709127276724404979 | 0.0


## 3. La courbe $y^2 = b(\chi)\,c(\chi)$ : racines et dégénérescences

$b$ a les racines $1$ et $-1/s$ ; $c$ (discriminant $(s+1)(s+9)$) les racines
$r_\pm = \frac{(3+s) \pm \sqrt{(s+1)(s+9)}}{2s}$. Quatre racines réelles
distinctes pour $s > 0$ : une quartique de genre 1 — **la courbe elliptique du
sunrise, en forme de Jacobi**. Deux dégénérescences la bornent :

- **$s \to 0$** : la racine $-1/s$ s'échappe à l'infini, la quartique perd un
  facteur — le genre tombe à 0. Le monde elliptique se réduit au monde du
  dilogarithme : l'ancre $I(0) = \frac{4}{\sqrt3}\mathrm{Cl}_2(\pi/3)$ vit
  **à la frontière**.
- **$s \to \infty$** : $r_- = -\frac1s + \frac{4}{s^2}$ et $-1/s$ — les deux
  racines négatives se collisionnent en $0$, l'écart entre elles tombant en
  $4/s^2$ : la pointe (cusp) de la famille.

In [3]:
def racines(s):
    # Quatre racines de la quartique, triees croissantes.
    r = [-1/s,
         1,
         ((3+s) - sqrt((s+1)*(s+9)))/(2*s),
         ((3+s) + sqrt((s+1)*(s+9)))/(2*s)]
    return sorted(r)

for s in [mpf('0.01'), 4, 9, 16, mpf('1e6')]:
    r = racines(s)
    print(f"s={mp.nstr(s, 6):>9} : racines = [{', '.join(mp.nstr(x, 8) for x in r)}]")
print()
print("s->0   : -1/s fuit a -infini (racine qui s'echappe)")
print("s->inf : -1/s et r_- se collisionnent en 0 (pointe)")

s=     0.01 : racines = [-100.0, -0.33186003, 1, 301.33186]
s=        4 : racines = [-0.25, -0.13278222, 1, 1.8827822]
s=        9 : racines = [-0.1111111111111111, -0.078689326, 1, 1.4120227]
s=       16 : racines = [-0.0625, -0.050485254, 1, 1.2379853]
s=   1.0e+6 : racines = [-1.0e-6, -9.99996e-7, 1, 1.000004]

s->0   : -1/s fuit a -infini (racine qui s'echappe)
s->inf : -1/s et r_- se collisionnent en 0 (pointe)


## 4. Les périodes : mesurées, puis réduites à $K(k)$

Avec $e_1 < e_2 < e_3 < e_4$ les racines triées, les deux cycles réels se
lisent sur les intervalles où $bc$ garde un signe : $bc > 0$ sur $(e_2, e_3)$
(c'est le domaine $\chi \in [0,1]$ de l'intégrale !) et $bc < 0$ sur
$(e_1, e_2)$. On mesure

$$\psi_1 = \int_{e_2}^{e_3} \frac{d\chi}{\sqrt{b\,c}}, \qquad
\psi_2 = \int_{e_1}^{e_2} \frac{d\chi}{\sqrt{-b\,c}},$$

mesurées sous la substitution $\chi = m + h\sin\theta$ (avec $m, h$ le milieu et
la demi-largeur de l'intervalle), qui donne exactement
$d\chi/\sqrt{(\chi-\mathrm{lo})(\mathrm{hi}-\chi)} = d\theta$ : la singularité
$1/\sqrt{\ }$ des deux bouts disparaît et l'intégrande devient lisse (facteur des
deux racines restantes, chacune rendue positive sur l'intervalle) — sans elle la
quadrature plafonne vers $10^{-9}$, ce qui a été mesuré avant d'être corrigé.

et la réduction de Legendre d'une quartique à quatre racines réelles relie les
deux au module $k^2 = \frac{(e_4-e_3)(e_2-e_1)}{(e_4-e_2)(e_3-e_1)}$ :

$$\frac{\psi_1}{\psi_2} = \frac{K(1-k^2)}{K(k^2)}.$$

La première écriture de cette section avait le rapport **inversé** — le planted
truth l'a attrapée ($1/0{,}4140 = 2{,}4156$ : c'est bien $\psi_1/\psi_2$ qui vaut
$K(1-k^2)/K(k^2)$), et c'est la version mesurée qui est commise ici. Le module
$k^2$ se mesure aussi **directement par birapport des racines** : deux voies
indépendantes pour le même nombre.

In [4]:
def periodes(s):
    # psi1 (reel, e2..e3) et psi2 (perpendiculaire, e1..e2) par quadrature.
    # Substitution chi = m + h*sin(theta) : dchi/sqrt((chi-lo)(hi-chi)) = dtheta,
    # la singularite 1/sqrt aux deux bouts disparait, l'integrande devient lisse.
    # Les deux racines restantes encadrent l'intervalle : chaque facteur (chi - o)
    # est rendu positif par un signe fixe au milieu.
    e = racines(s)
    def psi(lo, hi, autres):
        mid, half = (lo+hi)/2, (hi-lo)/2
        signes = [(o, 1 if mid > o else -1) for o in autres]
        def g(th):
            chi = mid + half*mp.sin(th)
            r = mpf(1)
            for o, sg in signes:
                r *= sg*(chi-o)
            return 1/sqrt(r)
        return quad(g, [-mp.pi/2, 0, mp.pi/2])
    return psi(e[1], e[2], [e[0], e[3]]), psi(e[0], e[1], [e[2], e[3]])

def k2_module(s):
    # Module de Legendre par birapport des racines.
    e = racines(s)
    return ((e[3]-e[2])*(e[1]-e[0])) / ((e[3]-e[1])*(e[2]-e[0]))

for s in [9, 16, 25]:
    psi1, psi2 = periodes(s)
    k2 = k2_module(s)
    ratio_mesure = mp.re(psi1/psi2)
    ratio_legendre = ellipk(1-k2)/ellipk(k2)
    print(f"s={s:3d} : k^2 = {mp.nstr(k2, 12)}")
    print(f"        psi1/psi2 mesure   = {mp.nstr(ratio_mesure, 20)}")
    print(f"        K(1-k2)/K(k2)      = {mp.nstr(ratio_legendre, 20)}")
    print(f"        ecart relatif      = {mp.nstr(abs(ratio_mesure-ratio_legendre)/ratio_legendre, 3)}")
    print()

s=  9 : k^2 = 0.00806504495005
        psi1/psi2 mesure   = 2.4155769875494845698
        K(1-k2)/K(k2)      = 2.4155769875494845858
        ecart relatif      = 6.61e-18

s= 16 : k^2 = 0.00208862860188
        psi1/psi2 mesure   = 2.8465788237216816068
        K(1-k2)/K(k2)      = 2.8465788237216816068
        ecart relatif      = 2.08e-30

s= 25 : k^2 = 0.00066981061043
        psi1/psi2 mesure   = 3.2088085228959175267
        K(1-k2)/K(k2)      = 3.2088085228959175373
        ecart relatif      = 3.31e-18



## 5. $\tau(s)$ court : la courbe dépend de l'échelle

Le rapport des périodes définit le paramètre modulaire
$\tau(s) = i\,\psi_1/\psi_2$. Dans les carnets 01 et 02, tout se réduisait à
des constantes fixées ($\mathrm{Cl}_2$ de quelques angles) ; ici **la courbe
elle-même dépend de $s$** — c'est la leçon du rung elliptique : les périodes
remplacent les constantes, et $\tau(s)$ n'est constant nulle part. Au carnet
02, l'hypothèse naïve $I(s) = A\,K(k)$ à module
$k^2 = (1-\sqrt{1-9/s})/2$ échouait déjà ($A$ refusait d'être constant) — la
mesure ci-dessous montre que **ce module naïf n'est pas non plus celui de la
quartique** : deux familles différentes, la mesure tranche.

In [5]:
print(f"{'s':>8} | {'k^2 quartique':>16} | {'k^2 naive (carnet 02)':>22} | {'tau = i psi1/psi2':>22}")
for s in [9, 16, 25, 100, mpf('1e4')]:
    k2q = k2_module(s)
    k2n = (1 - sqrt(1 - mpf(9)/s))/2
    psi1, psi2 = periodes(s)
    print(f"{mp.nstr(s, 6):>8} | {mp.nstr(k2q, 16):>16} | {mp.nstr(k2n, 22):>22} | {mp.nstr(mp.re(psi1/psi2), 22):>22}")
print()
# la pointe : k^2 ~ 16/s^3 (ecart 4/s^2 entre les racines qui collisionnent, fois e4-e3 ~ 4/s)
print("k^2 * s^3/16 (attendu -> 1) :")
for s in [mpf('1e4'), mpf('1e6'), mpf('1e8')]:
    print(f"  s={mp.nstr(s,4)} : {mp.nstr(k2_module(s)*s**3/16, 12)}")

       s |    k^2 quartique |  k^2 naive (carnet 02) |      tau = i psi1/psi2
       9 | 0.008065044950046265 |                    0.5 | 2.415576987549484569804
      16 | 0.002088628601881131 | 0.1692810861169261761873 | 2.846578823721681606773


      25 | 0.0006698106104295918 |                    0.1 | 3.20880852289591752668


     100 | 1.424739782031008e-5 | 0.02303039929152717542369 | 4.434539869325926435733
 10000.0 | 1.598081822369432e-11 | 0.0002250506477940725316837 | 8.795609024796386378584

k^2 * s^3/16 (attendu -> 1) :
  s=1.0e+4 : 0.998801138981
  s=1.0e+6 : 0.999988000114
  s=1.0e+8 : 0.99999988


## 6. Refus documentés (anneau déclaré avant recherche)

- **La forme fermée d'Adams–Bogner–Weinzierl** ($I$ en dilogarithmes
  elliptiques multiples) est **citée, pas reproduite** — la reproduire exigerait
  la construction du bidilogarithme elliptique, hors du périmètre d'un carnet
  de mesure. Notre ancre de fermeture reste $I(0)$.
- **Aucune constante fermée n'est identifiée pour $I(s)$, $s > 0$** : les
  sorties ci-dessus sont des mesures, pas des identifications. Chercher
  $I(9) = \alpha\,K(k) + \beta$ à $\alpha,\beta$ « simples » sans dérivation
  serait fabriquer un résultat.
- **Le module naïf du carnet 02 n'est pas celui de la quartique** (mesure §5) :
  c'est une question de corps de modules (courbes isogènes = modules reliés par
  transformations de Landen), pas une erreur à corriger dans l'un ou l'autre
  sens.

In [6]:
# Verification finale : la voie C reproduit l'ancre a s=0 par limite
for s in [mpf('0.1'), mpf('0.01'), mpf('0.001')]:
    c = I_C(s)
    print(f"I_C({mp.nstr(s,3)}) = {mp.nstr(c, 20)}   ancre = {mp.nstr(anchor, 20)}   ecart = {mp.nstr(abs(c-anchor)/anchor, 3)}")
print("(l'ecart decroit comme O(s) : la courbe degenere vers le monde Cl2 a la limite s=0)")

I_C(0.1) = 2.3325243164695720804   ancre = 2.3439072386894588906   ecart = 0.00486
I_C(0.01) = 2.3427616936616913665   ancre = 2.3439072386894588906   ecart = 0.000489
I_C(0.001) = 2.3437926110735361935   ancre = 2.3439072386894588906   ecart = 4.89e-5
(l'ecart decroit comme O(s) : la courbe degenere vers le monde Cl2 a la limite s=0)


## 7. Exercices

Trois exercices, à compléter — le carnet s'exécute de bout en bout sans eux
(règle C.1).

In [7]:
# Exercice 1. Rederiver la formule interne sur un autre couple (a, b).
# Verifier quad(lambda x: 1/(a + b*x*(1-x)), [0,1]) = 2*atanh(sqrt(b/c))/sqrt(b*c)
# avec c = b + 4a, pour a = 0.17, b = 2.3, a 25 chiffres.
# Etape 1 : poser a_exo, b_exo, c_exo = b_exo + 4*a_exo et comparer les deux cotes.
resultat_exo1 = None  # TODO etudiant
print("Exercice a completer : formule interne sur (0.17, 2.3)")

Exercice a completer : formule interne sur (0.17, 2.3)


In [8]:
# Exercice 2. Reproduire la reduction de Legendre a s = 36.
# Calculer psi1, psi2, k^2 par birapport, et verifier psi2/psi1 = K(1-k^2)/K(k^2).
# Etape 1 : appeler periodes(36) et k2_module(36), comparer au rapport des ellipk.
resultat_exo2 = None  # TODO etudiant
print("Exercice a completer : Legendre a s=36")

Exercice a completer : Legendre a s=36


In [9]:
# Exercice 3. La collision a l'infini : ou sont les racines a s = 1e8 ?
# Afficher racines(1e8) et verifier que les deux racines negatives tendent
# toutes deux vers -1/s, leur ecart e2 - e1 tombant en 4/s^2.
# Etape 1 : print(racines(mpf(10)**8)) puis comparer e2 - e1 a 4e-16.
resultat_exo3 = None  # TODO etudiant
print("Exercice a completer : collision des racines a s=1e8")

Exercice a completer : collision des racines a s=1e8


## 8. Ce que ce carnet établit

- **La courbe elliptique du sunrise exhibée par le calcul** : l'intégrale
  interne en $x$ est élémentaire (`artanh`), et le reste vit sur
  $y^2 = b\,c$ — quartique de genre 1, racines $\{1, -1/s, r_\pm\}$
  explicites, forme de Jacobi. Aucune courbe n'a été recopiée d'une référence.
- **$I(s)$ comme dilogarithme elliptique en formation** :
  $I(s) = 2\int_0^1 (1-\chi)\operatorname{artanh}\sqrt{b/c}\; d\chi/y$ —
  une intégrale de $\log$ contre la différentielle de première espèce, la
  construction même des polylogarithmes elliptiques.
- **Les périodes mesurées et réduites** : $\psi_2/\psi_1 = K(1-k^2)/K(k^2)$
  vérifié sur trois points, module $k^2$ mesuré par birapport des racines —
  deux voies indépendantes d'accord.
- **$\tau(s)$ non constant** : la courbe dépend de l'échelle. Les
  dégénérescences bornent la famille : $s \to 0$ (racine qui s'échappe —
  retour au monde $\mathrm{Cl}_2$, ancre $I(0)$), $s \to \infty$ (collision
  de racines — pointe, $k^2 \sim 16/s^3$).

L'étage suivant du classement BootLoops (K3, Calabi–Yau) demanderait des
graphes plus lourds (sunset à trois boucles) — hors périmètre de la série
tel qu'arbitré sur #20428.